In [ ]:
!pip install -q pypdf langchain-text-splitters sentence-transformers google-genai

In [ ]:
import os
import re
import urllib.request
import numpy as np

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from google.colab import userdata
from google import genai


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

URL_PDF = (
    "https://raw.githubusercontent.com/"
    "kimberlyn05/sales-knowledge-ai-agent/"
    "main/data/documents/Politicas_Internas_NexaShip.pdf"
)

RUTA_PDF = "/content/Politicas_Internas_NexaShip.pdf"

MODELO_EMBEDDINGS = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)


# ============================================================
# 2. DESCARGAR PDF
# ============================================================

if not os.path.exists(RUTA_PDF):
    urllib.request.urlretrieve(URL_PDF, RUTA_PDF)
    print("✓ PDF descargado desde GitHub")
else:
    print("✓ PDF disponible")


# ============================================================
# 3. EXTRAER TEXTO POR PÁGINA
# ============================================================

lector = PdfReader(RUTA_PDF)

paginas = []

for numero_pagina, pagina in enumerate(lector.pages, start=1):
    texto = pagina.extract_text() or ""

    paginas.append({
        "pagina": numero_pagina,
        "texto": texto
    })

print(f"✓ Páginas procesadas: {len(paginas)}")


# ============================================================
# 4. LIMPIAR TEXTO
# ============================================================

def limpiar_texto(texto):
    texto = texto.replace("\x7f", " ")
    texto = texto.replace("\u00a0", " ")

    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    return texto.strip()


paginas_limpias = []

for pagina in paginas:
    paginas_limpias.append({
        "pagina": pagina["pagina"],
        "texto": limpiar_texto(pagina["texto"])
    })

print("✓ Texto limpiado")


# ============================================================
# 5. CREAR CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_limpios = []
chunk_id = 1

for pagina in paginas_limpias:
    fragmentos = text_splitter.split_text(
        pagina["texto"]
    )

    for fragmento in fragmentos:
        chunks_limpios.append({
            "chunk_id": chunk_id,
            "pagina": pagina["pagina"],
            "texto": fragmento
        })

        chunk_id += 1

print(f"✓ Chunks creados: {len(chunks_limpios)}")


# ============================================================
# 6. CARGAR MODELO DE EMBEDDINGS
# ============================================================

modelo_embeddings = SentenceTransformer(
    MODELO_EMBEDDINGS
)

textos_chunks = [
    chunk["texto"]
    for chunk in chunks_limpios
]

embeddings_chunks = modelo_embeddings.encode(
    textos_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    f"✓ Embeddings generados: "
    f"{len(embeddings_chunks)}"
)


# ============================================================
# 7. CONFIGURAR GEMINI
# ============================================================

gemini_api_key = userdata.get(
    "GEMINI_API_KEY"
)

cliente_gemini = genai.Client(
    api_key=gemini_api_key
)

print("✓ Cliente Gemini configurado")


# ============================================================
# 8. FUNCIÓN DE BÚSQUEDA SEMÁNTICA
# ============================================================

def buscar_evidencia(pregunta, top_k=5):

    embedding_pregunta = modelo_embeddings.encode(
        [pregunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similitudes = (
        embeddings_chunks
        @ embedding_pregunta
    )

    indices_relevantes = np.argsort(
        similitudes
    )[::-1][:top_k]

    resultados = []

    for indice in indices_relevantes:
        chunk = chunks_limpios[indice]

        resultados.append({
            "chunk_id": chunk["chunk_id"],
            "pagina": chunk["pagina"],
            "similitud": float(
                similitudes[indice]
            ),
            "texto": chunk["texto"]
        })

    return resultados


print("✓ Búsqueda semántica preparada")


# ============================================================
# 9. FUNCIÓN DEL AGENTE
# ============================================================

def responder_con_evidencia(
    pregunta,
    top_k=5
):

    evidencias = buscar_evidencia(
        pregunta,
        top_k=top_k
    )

    contexto = "\n\n".join(
        [
            (
                f"[Fuente: página "
                f"{evidencia['pagina']}]\n"
                f"{evidencia['texto']}"
            )
            for evidencia in evidencias
        ]
    )

    prompt = f"""
Eres Sales Knowledge AI Agent,
un asistente interno de NexaShip.

Tu tarea es responder preguntas
utilizando exclusivamente la evidencia
documental proporcionada.

Reglas obligatorias:
- No inventes información.
- No utilices conocimiento externo.
- No completes datos faltantes mediante suposiciones.
- Si la evidencia no permite responder con seguridad,
  indícalo claramente.
- Responde en español.
- Sé claro y profesional.
- Menciona las páginas utilizadas como fuente.
- Si existen varias reglas relevantes,
  intégralas en una sola respuesta coherente.

PREGUNTA:
{pregunta}

EVIDENCIA DOCUMENTAL:
{contexto}

RESPUESTA:
"""

    respuesta = (
        cliente_gemini.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
    )

    return {
        "pregunta": pregunta,
        "respuesta": respuesta.text,
        "evidencias": evidencias
    }


print("✓ Agente configurado")
print()
print("🚀 Sales Knowledge AI Agent listo")

In [ ]:
# ============================================================
# DIAGNÓSTICO DE SKILLS Y ESCALAMIENTO
# ============================================================

terminos_escalamiento = [
    "skill",
    "especialidad",
    "escalamiento",
    "escalar"
]

for pagina in paginas_limpias:

    texto_minusculas = (
        pagina["texto"].lower()
    )

    if any(
        termino in texto_minusculas
        for termino in terminos_escalamiento
    ):
        print(
            f"\n{'=' * 100}"
        )
        print(
            f"PÁGINA {pagina['pagina']}"
        )
        print(
            "=" * 100
        )
        print(
            pagina["texto"]
        )

In [ ]:
import re
import unicodedata

STOPWORDS = {
    "a", "al", "ante", "como", "con", "cual", "cuando",
    "de", "del", "el", "en", "es", "esta", "este",
    "la", "las", "le", "lo", "los", "me", "mi",
    "no", "ofrece", "para", "por", "puede", "puedo",
    "que", "se", "si", "su", "sus", "un", "una",
    "y", "ya"
}


def normalizar_texto(texto):
    texto = texto.lower()

    texto = unicodedata.normalize(
        "NFD",
        texto
    )

    texto = "".join(
        caracter
        for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    return texto


def extraer_terminos_clave(pregunta):

    pregunta_normalizada = normalizar_texto(
        pregunta
    )

    palabras = re.findall(
        r"\b[a-z0-9]+\b",
        pregunta_normalizada
    )

    return [
        palabra
        for palabra in palabras
        if palabra not in STOPWORDS
        and len(palabra) >= 4
        and palabra != "nexaship"
    ]


def calcular_cobertura_lexica(
    pregunta,
    evidencias
):

    terminos = extraer_terminos_clave(
        pregunta
    )

    texto_evidencias = normalizar_texto(
        " ".join(
            evidencia["texto"]
            for evidencia in evidencias
        )
    )

    encontrados = [
        termino
        for termino in terminos
        if termino in texto_evidencias
    ]

    cobertura = (
        len(encontrados) / len(terminos)
        if terminos
        else 0.0
    )

    return {
        "terminos": terminos,
        "encontrados": encontrados,
        "cobertura": cobertura
    }


print("✓ Validador léxico configurado")

In [ ]:
def clasificar_evidencia(
    pregunta,
    evidencias
):
    if not evidencias:
        return {
            "estado": "insuficiente",
            "mejor_score": 0.0,
            "cobertura_lexica": 0.0
        }

    mejor_score = evidencias[0]["similitud"]

    resultado_lexico = calcular_cobertura_lexica(
        pregunta,
        evidencias
    )

    cobertura_lexica = (
        resultado_lexico["cobertura"]
    )

    # Sin coincidencia léxica relevante:
    # bloquear aunque exista similitud semántica
    if cobertura_lexica == 0.0:
        estado = "insuficiente"

    # Evidencia fuerte:
    # requiere similitud semántica alta
    # y cobertura léxica mínima
    elif (
        mejor_score >= 0.45
        and cobertura_lexica >= 0.20
    ):
        estado = "fuerte"

    # Score semántico alto pero cobertura muy baja:
    # no confiar automáticamente en el embedding
    elif (
        mejor_score >= 0.45
        and cobertura_lexica < 0.20
    ):
        estado = "insuficiente"

    # Evidencia intermedia
    elif mejor_score >= 0.30:
        estado = "incierta"

    else:
        estado = "insuficiente"

    return {
        "estado": estado,
        "mejor_score": mejor_score,
        "cobertura_lexica": cobertura_lexica
    }


print("✓ Clasificador de evidencia V4 configurado")

In [ ]:
# ============================================================
# CLASIFICACIÓN DE CONSULTAS POR SKILL
# ============================================================

SKILLS_NEXASHIP = {
    "Integraciones": [
        "api",
        "webhook",
        "webhooks",
        "ecommerce",
        "erp",
        "sincronizacion",
        "autenticacion",
        "integracion",
        "integraciones",
        "desarrollo personalizado",
        "desarrollos personalizados"
    ],

    "Operaciones Logísticas": [
        "cobertura",
        "estado de envio",
        "estados de envio",
        "operacion",
        "incidencia logistica",
        "incidencias logisticas",
        "restriccion",
        "restricciones",
        "transportista",
        "transportistas"
    ],

    "Comercial": [
        "comercial",
        "negociacion",
        "negociaciones",
        "descuento",
        "descuentos",
        "propuesta",
        "propuestas",
        "condicion especial",
        "condiciones especiales",
        "aprobacion comercial",
        "aprobaciones comerciales"
    ],

    "Facturación": [
        "factura",
        "facturas",
        "cobro",
        "cobros",
        "ajuste",
        "ajustes",
        "conciliacion",
        "conciliaciones",
        "discrepancia economica",
        "discrepancias economicas"
    ],

    "Contra Entrega": [
        "recaudo",
        "liquidacion",
        "pago contra entrega",
        "pagos contra entrega",
        "contra entrega"
    ],

    "Enterprise": [
        "alto volumen",
        "necesidad compleja",
        "necesidades complejas",
        "condicion corporativa",
        "condiciones corporativas",
        "multiples areas",
        "enterprise"
    ],

    "Protección de Datos": [
        "tratamiento de informacion",
        "exposicion accidental",
        "datos",
        "dato personal",
        "datos personales",
        "uso autorizado",
        "acceso no autorizado",
        "informacion confidencial"
    ]
}


def clasificar_skill(pregunta):

    pregunta_normalizada = normalizar_texto(
        pregunta
    )

    puntuaciones = {}

    for skill, terminos in SKILLS_NEXASHIP.items():

        coincidencias = []

        for termino in terminos:

            termino_normalizado = normalizar_texto(
                termino
            )

            if termino_normalizado in pregunta_normalizada:
                coincidencias.append(
                    termino
                )

        puntuaciones[skill] = {
            "score": len(coincidencias),
            "coincidencias": coincidencias
        }

    skill_principal = max(
        puntuaciones,
        key=lambda skill: puntuaciones[skill]["score"]
    )

    mejor_score = (
        puntuaciones[skill_principal]["score"]
    )

    if mejor_score == 0:
        return {
            "skill": None,
            "coincidencias": [],
            "puntuaciones": puntuaciones
        }

    return {
        "skill": skill_principal,
        "coincidencias": (
            puntuaciones[skill_principal]["coincidencias"]
        ),
        "puntuaciones": puntuaciones
    }


print("✓ Clasificador por skill configurado")

In [ ]:
preguntas_skills = [
    "¿Puedo ofrecer una integración personalizada con un ERP?",
    "¿Qué hago si un transportista presenta una incidencia logística?",
    "¿Puedo ofrecer un descuento especial al cliente?",
    "¿Cómo resuelvo una discrepancia en una factura?",
    "¿Qué condiciones aplican para pagos contra entrega?",
    "¿Cómo manejo un cliente enterprise con necesidades complejas?",
    "¿Qué hago ante una exposición accidental de datos personales?",
    "¿Cuál es el menú de la cafetería?"
]


for pregunta in preguntas_skills:

    resultado = clasificar_skill(
        pregunta
    )

    print(f"PREGUNTA: {pregunta}")
    print(f"SKILL: {resultado['skill']}")
    print(
        f"COINCIDENCIAS: "
        f"{resultado['coincidencias']}"
    )
    print("-" * 100)

In [ ]:
# ============================================================
# CLASIFICACIÓN DE NIVEL DE ESCALAMIENTO
# ============================================================

TERMINOS_EXCEPCION = [
    "excepcion",
    "no estandar",
    "fuera de politica",
    "riesgo relevante",
    "requiere aprobacion",
    "aprobacion especial"
]

TERMINOS_VALIDACION_ESPECIALIZADA = [
    "validacion tecnica",
    "validacion especializada",
    "requiere validacion",
    "consultar especialista",
    "consulta especializada"
]

TERMINOS_MULTIDISCIPLINARIOS = [
    "integracion personalizada",
    "condicion comercial excepcional",
    "cliente enterprise",
    "alto volumen",
    "impacto financiero y tecnico",
    "multiples areas",
    "varias areas",
    "requerimiento operativo no estandar"
]


def clasificar_escalamiento(
    pregunta,
    estado_evidencia
):

    pregunta_normalizada = normalizar_texto(
        pregunta
    )

    resultado_skill = clasificar_skill(
        pregunta
    )

    skill = resultado_skill["skill"]

    coincidencias_excepcion = [
        termino
        for termino in TERMINOS_EXCEPCION
        if normalizar_texto(termino)
        in pregunta_normalizada
    ]

    coincidencias_validacion = [
        termino
        for termino in TERMINOS_VALIDACION_ESPECIALIZADA
        if normalizar_texto(termino)
        in pregunta_normalizada
    ]

    coincidencias_multi = [
        termino
        for termino in TERMINOS_MULTIDISCIPLINARIOS
        if normalizar_texto(termino)
        in pregunta_normalizada
    ]

     # Nivel 3 — Revisión multidisciplinaria
    if coincidencias_multi:
        return {
            "nivel": 3,
            "tipo": "Revisión multidisciplinaria",
            "skill": skill,
            "motivo": (
                "La consulta presenta condiciones "
                "que pueden involucrar varias áreas."
            ),
            "coincidencias": coincidencias_multi
        }

    # Nivel 2 — Líder del área
    if coincidencias_excepcion:
        return {
            "nivel": 2,
            "tipo": "Líder del área",
            "skill": skill,
            "motivo": (
                "La consulta implica una excepción, "
                "condición no estándar o aprobación especial."
            ),
            "coincidencias": coincidencias_excepcion
        }

    # Nivel 1 — Validación especializada explícita
    if coincidencias_validacion and skill is not None:
        return {
            "nivel": 1,
            "tipo": "Especialista de la skill",
            "skill": skill,
            "motivo": (
                "La consulta solicita explícitamente "
                "validación especializada."
            ),
            "coincidencias": coincidencias_validacion
        }

    # Nivel 0 — Resolución documental
    if estado_evidencia == "fuerte":
        return {
            "nivel": 0,
            "tipo": "Resolución documental",
            "skill": skill,
            "motivo": (
                "La evidencia documental permite "
                "resolver la consulta directamente."
            ),
            "coincidencias": []
        }

    # Nivel 1 — Especialista de la skill
    if skill is not None:
        return {
            "nivel": 1,
            "tipo": "Especialista de la skill",
            "skill": skill,
            "motivo": (
                "La consulta requiere validación "
                "especializada dentro de una skill."
            ),
            "coincidencias": []
        }

    return {
        "nivel": None,
        "tipo": "Sin ruta determinada",
        "skill": None,
        "motivo": (
            "No existe evidencia ni una skill "
            "suficiente para determinar una ruta segura."
        ),
        "coincidencias": []
    }

print("✓ Clasificador de escalamiento configurado")

In [ ]:
casos_escalamiento = [
    {
        "pregunta": (
            "¿Qué servicios puede presentar "
            "el equipo comercial como disponibles?"
        ),
        "estado_evidencia": "fuerte"
    },
    {
        "pregunta": (
            "Tengo una consulta sobre una API "
            "que requiere validación técnica"
        ),
        "estado_evidencia": "incierta"
    },
    {
        "pregunta": (
            "¿Qué hago ante una excepción comercial "
            "que requiere aprobación especial?"
        ),
        "estado_evidencia": "incierta"
    },
    {
        "pregunta": (
            "Un cliente enterprise solicita una "
            "integración personalizada con una "
            "condición comercial excepcional"
        ),
        "estado_evidencia": "incierta"
    },
    {
        "pregunta": (
            "¿Cuál es el menú de la cafetería?"
        ),
        "estado_evidencia": "insuficiente"
    }
]


for caso in casos_escalamiento:

    resultado = clasificar_escalamiento(
        caso["pregunta"],
        caso["estado_evidencia"]
    )

    print(
        f"PREGUNTA: {caso['pregunta']}"
    )
    print(
        f"ESTADO EVIDENCIA: "
        f"{caso['estado_evidencia']}"
    )
    print(
        f"NIVEL: {resultado['nivel']}"
    )
    print(
        f"TIPO: {resultado['tipo']}"
    )
    print(
        f"SKILL: {resultado['skill']}"
    )
    print(
        f"MOTIVO: {resultado['motivo']}"
    )
    print(
        f"COINCIDENCIAS: "
        f"{resultado['coincidencias']}"
    )
    print("-" * 100)

In [ ]:
!pip install -q groq

In [ ]:
from google.colab import userdata
from groq import Groq

# Obtener la API key desde los Secrets de Colab
groq_api_key = userdata.get("GROQ_API_KEY")

# Crear cliente de Groq
cliente_groq = Groq(
    api_key=groq_api_key
)

print("✓ Cliente Groq configurado correctamente")

In [ ]:
def generar_respuesta_llm(prompt):
    """
    Intenta generar la respuesta con Gemini.
    Si Gemini falla, utiliza Groq como fallback.
    """

    try:
        # Proveedor principal: Gemini
        respuesta_gemini = cliente_gemini.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return {
            "texto": respuesta_gemini.text,
            "proveedor": "gemini",
            "fallback_usado": False
        }

    except Exception as error_gemini:
        print(
            "⚠ Gemini no disponible. "
            "Intentando con Groq..."
        )

        try:
            # Proveedor alternativo: Groq
            respuesta_groq = cliente_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )

            return {
                "texto": respuesta_groq.choices[0].message.content,
                "proveedor": "groq",
                "fallback_usado": True
            }

        except Exception as error_groq:
            return {
                "texto": (
                    "No fue posible generar una respuesta "
                    "en este momento porque los proveedores "
                    "de lenguaje no están disponibles."
                ),
                "proveedor": None,
                "fallback_usado": True,
                "error_gemini": str(error_gemini),
                "error_groq": str(error_groq)
            }


print("✓ Fallback Gemini → Groq configurado")

In [ ]:
def responder_agente_controlado(
    pregunta,
    top_k=5
):
    # 1. Recuperar evidencia
    evidencias = buscar_evidencia(
        pregunta,
        top_k=top_k
    )

    # 2. Clasificar evidencia
    clasificacion = clasificar_evidencia(
        pregunta,
        evidencias
    )

    estado = clasificacion["estado"]
    mejor_score = clasificacion["mejor_score"]

    # 3. Clasificar ruta de escalamiento
    escalamiento = clasificar_escalamiento(
        pregunta,
        estado
    )

    # 4. Bloquear consultas claramente sin evidencia
    # Aquí NO se llama a ningún LLM
    if estado == "insuficiente":
        return {
            "pregunta": pregunta,
            "respuesta": (
                "No encontré evidencia documental suficiente "
                "en las fuentes disponibles para responder "
                "esta consulta con seguridad."
            ),
            "estado_evidencia": estado,
            "mejor_score": mejor_score,
            "nivel_escalamiento": escalamiento["nivel"],
            "tipo_escalamiento": escalamiento["tipo"],
            "skill": escalamiento["skill"],
            "motivo_escalamiento": escalamiento["motivo"],
            "evidencias": evidencias,
            "llamada_llm": False,
            "proveedor": None,
            "fallback_usado": False
        }

    # 5. Construir contexto documental
    contexto = "\n\n".join(
        [
            (
                f"[Fuente: página "
                f"{evidencia['pagina']}]\n"
                f"{evidencia['texto']}"
            )
            for evidencia in evidencias
        ]
    )

    # 6. Instrucciones según nivel de evidencia
    if estado == "incierta":
        instruccion_evidencia = """
La relevancia de la evidencia recuperada es incierta.

Debes ser especialmente conservador:
- Responde solo si la evidencia contiene información
  directamente aplicable a la pregunta.
- No deduzcas reglas que no estén expresadas.
- No conviertas similitudes temáticas en hechos.
- Si la evidencia no responde directamente,
  indica que no existe información suficiente.
"""
    else:
        instruccion_evidencia = """
La evidencia recuperada presenta una relevancia fuerte.

Aun así:
- Responde exclusivamente con base en la evidencia.
- No agregues información externa.
- No realices suposiciones no respaldadas.
"""

    # 7. Construir prompt
    prompt = f"""
Eres Sales Knowledge AI Agent,
un asistente interno de NexaShip.

Tu tarea es responder preguntas utilizando
exclusivamente la evidencia documental proporcionada.

Reglas obligatorias:
- No inventes información.
- No utilices conocimiento externo.
- No completes datos faltantes mediante suposiciones.
- Si la evidencia no permite responder con seguridad,
  indícalo claramente.
- Responde en español.
- Sé claro y profesional.
- Menciona las páginas utilizadas como fuente.

ESTADO DE EVIDENCIA:
{estado}

INSTRUCCIONES ADICIONALES:
{instruccion_evidencia}

PREGUNTA:
{pregunta}

EVIDENCIA DOCUMENTAL:
{contexto}

RESPUESTA:
"""

    # 8. Generar respuesta con fallback automático
    resultado_llm = generar_respuesta_llm(
        prompt
    )

    # 9. Retornar resultado completo
    return {
        "pregunta": pregunta,
        "respuesta": resultado_llm["texto"],
        "estado_evidencia": estado,
        "mejor_score": mejor_score,
        "nivel_escalamiento": escalamiento["nivel"],
        "tipo_escalamiento": escalamiento["tipo"],
        "skill": escalamiento["skill"],
        "motivo_escalamiento": escalamiento["motivo"],
        "evidencias": evidencias,
        "llamada_llm": True,
        "proveedor": resultado_llm["proveedor"],
        "fallback_usado": resultado_llm["fallback_usado"]
    }


print("✓ Agente controlado con fallback y escalamiento configurado")

In [ ]:
# ============================================================
# PRUEBA DE INTEGRACIÓN DEL ESCALAMIENTO EN EL AGENTE
# ============================================================

preguntas_integracion_escalamiento = [
    "¿Qué servicios puede presentar el equipo comercial como disponibles?",
    "Tengo una consulta sobre una API que requiere validación técnica",
    "¿Qué hago ante una excepción comercial que requiere aprobación especial?",
    (
        "Un cliente enterprise solicita una integración personalizada "
        "con una condición comercial excepcional"
    ),
    "¿Cuál es el menú de la cafetería?"
]


for pregunta in preguntas_integracion_escalamiento:

    resultado = responder_agente_controlado(
        pregunta
    )

    print("=" * 100)
    print(f"PREGUNTA: {resultado['pregunta']}")
    print(f"ESTADO EVIDENCIA: {resultado['estado_evidencia']}")
    print(f"MEJOR SCORE: {resultado['mejor_score']:.4f}")
    print(f"NIVEL ESCALAMIENTO: {resultado['nivel_escalamiento']}")
    print(f"TIPO ESCALAMIENTO: {resultado['tipo_escalamiento']}")
    print(f"SKILL: {resultado['skill']}")
    print(f"MOTIVO: {resultado['motivo_escalamiento']}")
    print(f"¿SE LLAMÓ AL LLM?: {resultado['llamada_llm']}")
    print(f"PROVEEDOR: {resultado['proveedor']}")
    print()

In [ ]:
casos_evaluacion = [
    {
        "id": "EVAL-001",
        "pregunta": "¿Puedo prometerle a un cliente una integración personalizada?",
        "categoria": "comercial",
        "esperado_evidencia": True
    },
    {
        "id": "EVAL-002",
        "pregunta": "¿Qué debo hacer si no tengo evidencia suficiente?",
        "categoria": "politica",
        "esperado_evidencia": True
    },
    {
        "id": "EVAL-003",
        "pregunta": "¿Cómo debo escalar una consulta según la skill?",
        "categoria": "escalamiento",
        "esperado_evidencia": True
    },
    {
        "id": "EVAL-004",
        "pregunta": "¿Cuántos días de vacaciones tienen los empleados de NexaShip?",
        "categoria": "fuera_documento",
        "esperado_evidencia": False
    },
    {
        "id": "EVAL-005",
        "pregunta": "¿Cuál es el menú de la cafetería de NexaShip?",
        "categoria": "fuera_documento",
        "esperado_evidencia": False
    }
]

print(f"✓ Dataset de evaluación creado: {len(casos_evaluacion)} casos")

In [ ]:
casos_validacion = [
    {
        "id": "VAL-001",
        "pregunta": "¿Qué servicios puede presentar el equipo comercial como disponibles?",
        "esperado_evidencia": True
    },
    {
        "id": "VAL-002",
        "pregunta": "¿Qué debe hacer una persona comercial ante un requerimiento técnico no documentado?",
        "esperado_evidencia": True
    },
    {
        "id": "VAL-003",
        "pregunta": "¿Se puede compartir información interna de aprobaciones con un cliente?",
        "esperado_evidencia": True
    },
    {
        "id": "VAL-004",
        "pregunta": "¿Cuál es el salario promedio de los empleados de NexaShip?",
        "esperado_evidencia": False
    },
    {
        "id": "VAL-005",
        "pregunta": "¿NexaShip ofrece seguro médico a sus empleados?",
        "esperado_evidencia": False
    }
]

print(
    f"✓ Dataset de validación creado: "
    f"{len(casos_validacion)} casos"
)

In [ ]:
# ============================================================
# CASOS ADVERSARIALES DE REGRESIÓN
# ============================================================

casos_adversariales = [
    {
        "id": "ADV-001",
        "pregunta": (
            "¿NexaShip ofrece seguro médico privado a sus empleados "
            "y qué porcentaje de la prima cubre la empresa?"
        ),
        "esperado_evidencia": False,
        "motivo": (
            "Controlar falso positivo semántico causado por "
            "la ambigüedad del término cobertura."
        )
    }
]

print(
    f"✓ Casos adversariales creados: "
    f"{len(casos_adversariales)}"
)

In [ ]:
# ============================================================
# EVALUACIÓN FINAL V4 + CASOS ADVERSARIALES
# ============================================================

todos_los_casos_v4 = (
    casos_evaluacion
    + casos_validacion
    + casos_adversariales
)

aciertos = 0

for caso in todos_los_casos_v4:

    evidencias = buscar_evidencia(
        caso["pregunta"],
        top_k=5
    )

    clasificacion = clasificar_evidencia(
        caso["pregunta"],
        evidencias
    )

    estado = clasificacion["estado"]

    predice_evidencia = (
        estado != "insuficiente"
    )

    es_acierto = (
        predice_evidencia
        == caso["esperado_evidencia"]
    )

    if es_acierto:
        aciertos += 1

    print(f"CASO: {caso['id']}")
    print(f"Pregunta: {caso['pregunta']}")
    print(
        f"Esperado evidencia: "
        f"{caso['esperado_evidencia']}"
    )
    print(f"Estado: {estado}")
    print(
        f"Score: "
        f"{clasificacion['mejor_score']:.4f}"
    )
    print(
        f"Cobertura léxica: "
        f"{clasificacion['cobertura_lexica']:.4f}"
    )
    print(
        "Resultado: "
        + (
            "✓ ACIERTO"
            if es_acierto
            else "✗ ERROR"
        )
    )
    print("-" * 80)


precision = (
    aciertos / len(todos_los_casos_v4)
)

print()
print(
    "EVALUACIÓN FINAL V4 "
    + "CON REGRESIÓN ADVERSARIAL"
)
print("=" * 50)
print(
    f"Total de casos: "
    f"{len(todos_los_casos_v4)}"
)
print(
    f"Aciertos: "
    f"{aciertos}/{len(todos_los_casos_v4)}"
)
print(
    f"Precisión: "
    f"{precision:.2%}"
)

In [ ]:
preguntas_prueba_v3 = [
    "¿Cuántos días de vacaciones tienen los empleados de NexaShip?",
    "¿NexaShip ofrece seguro médico a sus empleados?"
]

for pregunta in preguntas_prueba_v3:

    resultado = responder_agente_controlado(
        pregunta
    )

    print("PREGUNTA:")
    print(resultado["pregunta"])

    print("\nRESPUESTA:")
    print(resultado["respuesta"])

    print("\nESTADO DE EVIDENCIA:")
    print(resultado["estado_evidencia"])

    print("\nMEJOR SCORE:")
    print(f"{resultado['mejor_score']:.4f}")

    print("\n¿SE LLAMÓ AL LLM?")
    print(resultado["llamada_llm"])

    print("\nPROVEEDOR:")
    print(resultado["proveedor"])

    print("\n¿SE USÓ FALLBACK?")
    print(resultado["fallback_usado"])

    print("\n" + "=" * 100 + "\n")

In [ ]:
todos_los_casos = (
    casos_evaluacion
    + casos_validacion
)

aciertos = 0

for caso in todos_los_casos:

    evidencias = buscar_evidencia(
        caso["pregunta"],
        top_k=5
    )

    clasificacion = clasificar_evidencia(
        caso["pregunta"],
        evidencias
    )

    evidencia_detectada = (
        clasificacion["estado"]
        != "insuficiente"
    )

    acierto = (
        evidencia_detectada
        == caso["esperado_evidencia"]
    )

    if acierto:
        aciertos += 1

    print(f"CASO: {caso['id']}")
    print(f"Pregunta: {caso['pregunta']}")
    print(
        f"Esperado evidencia: "
        f"{caso['esperado_evidencia']}"
    )
    print(
        f"Estado: "
        f"{clasificacion['estado']}"
    )
    print(
        f"Score: "
        f"{clasificacion['mejor_score']:.4f}"
    )
    print(
        f"Cobertura léxica: "
        f"{clasificacion['cobertura_lexica']:.4f}"
    )
    print(
        f"Resultado: "
        f"{'✓ ACIERTO' if acierto else '✗ ERROR'}"
    )
    print("-" * 80)


total = len(todos_los_casos)

precision = (
    aciertos / total * 100
)

print()
print("EVALUACIÓN FINAL DEL CLASIFICADOR")
print("=" * 50)
print(f"Total de casos: {total}")
print(f"Aciertos: {aciertos}/{total}")
print(f"Precisión: {precision:.2f}%")

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr


def consultar_agente_interfaz(pregunta):

    # Evitar consultas vacías
    if not pregunta or not pregunta.strip():
        return (
            "Escribe una pregunta para consultar el agente.",
            "-",
            "-",
            "-",
            "-",
            "-",
            "-",
            "-",
            "-"
        )

    # Ejecutar el agente real
    resultado = responder_agente_controlado(
        pregunta.strip()
    )

    # Formatear score
    mejor_score = resultado.get(
        "mejor_score"
    )

    score_formateado = (
        f"{mejor_score:.4f}"
        if mejor_score is not None
        else "-"
    )

    # Formatear valores opcionales
    skill = resultado.get("skill") or "-"
    nivel = resultado.get("nivel_escalamiento")
    tipo = resultado.get("tipo_escalamiento") or "-"
    motivo = resultado.get("motivo_escalamiento") or "-"
    proveedor = resultado.get("proveedor") or "-"

    nivel_formateado = (
        str(nivel)
        if nivel is not None
        else "-"
    )

    fallback = (
        "Sí"
        if resultado.get("fallback_usado")
        else "No"
    )

    return (
        resultado.get("respuesta", ""),
        resultado.get("estado_evidencia", "-"),
        score_formateado,
        skill,
        nivel_formateado,
        tipo,
        motivo,
        proveedor,
        fallback
    )


print("✓ Función puente con motivo de escalamiento configurada")

In [ ]:
# ============================================================
# INTERFAZ MÍNIMA CON GRADIO
# ============================================================

with gr.Blocks(
    title="Sales Knowledge AI Agent"
) as demo:

    gr.Markdown(
        """
        # Sales Knowledge AI Agent

        Consulta conocimiento interno de NexaShip
        mediante preguntas en lenguaje natural.

        Las respuestas se generan con base en
        evidencia documental disponible.
        """
    )

    pregunta_input = gr.Textbox(
        label="Pregunta",
        placeholder=(
            "Ejemplo: ¿Puedo ofrecer una "
            "integración personalizada?"
        ),
        lines=3
    )

    boton_consultar = gr.Button(
        "Consultar agente",
        variant="primary"
    )

    respuesta_output = gr.Textbox(
        label="Respuesta",
        lines=8
    )

    gr.Markdown(
        "## Trazabilidad de la consulta"
    )

    with gr.Row():
        estado_output = gr.Textbox(
            label="Estado de evidencia"
        )

        score_output = gr.Textbox(
            label="Mejor score"
        )

        skill_output = gr.Textbox(
            label="Skill"
        )

    with gr.Row():
        nivel_output = gr.Textbox(
            label="Nivel de escalamiento"
        )

        tipo_output = gr.Textbox(
            label="Tipo de escalamiento"
        )

    motivo_output = gr.Textbox(
        label="Motivo del escalamiento",
        lines=2
    )

    with gr.Row():
        proveedor_output = gr.Textbox(
            label="Proveedor LLM"
        )

        fallback_output = gr.Textbox(
            label="Fallback usado"
        )

    boton_consultar.click(
        fn=consultar_agente_interfaz,
        inputs=pregunta_input,
        outputs=[
            respuesta_output,
            estado_output,
            score_output,
            skill_output,
            nivel_output,
            tipo_output,
            motivo_output,
            proveedor_output,
            fallback_output
        ]
    )


print("✓ Interfaz Gradio con motivo de escalamiento configurada")

In [ ]:
# ============================================================
# LANZAR INTERFAZ
# ============================================================

demo.launch(
    share=True,
    debug=False
)